In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and installs required packages.
import os, sys

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in student_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/student_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# lightgbm: ML model backend
# mlforecast: lag transforms, cross-validation, PredictionIntervals
os.system("pip install -q lightgbm mlforecast")

print(f"✓ Setup complete — {os.getcwd()}")

# Module 5 — ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---
## 5.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.utils import PredictionIntervals

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
    USE_INTERVALS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast
from src.workshop_utils import get_micro_subset

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")
print(f"USE_INTERVALS    = {USE_INTERVALS}")

---
## 5.2 — Load Panel and Micro Subset
**[Watch Only]**

In [ ]:
panel = load_checkpoint("03_validated_panel")
micro, top_series = get_micro_subset(panel, n=MICRO_SUBSET_N)

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 5.3 — Why ML Forecasting?
**[Watch Only]**

AutoETS is a strong univariate model. It sees one series at a time. It cannot learn patterns that span series — shared seasonal profiles, cross-SKU promotional lifts, or price elasticity signals.

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously. Each series becomes a set of training rows. The model learns patterns across the entire category portfolio.

**The compute tradeoff is real:**  
AutoETS on 1,000 series takes seconds. LightGBM on 1,000 series with lag features takes longer and produces a model artifact you now have to version, deploy, and monitor. The accuracy gain must justify that operational cost.

**The three feature configurations we will build (sections 5.5A–5.5C):**

| Section | Feature type | What it adds |
|---|---|---|
| **5.5A — Base** | Lags (7, 14, 28), rolling mean, day-of-week + month | Demand history and calendar |
| **5.5B — Rich** | Extended lags (7–28), rolling stats, Fourier features, seasonal rolling | More complete numeric/date signal |
| **5.5C — Categorical** | M5 hierarchy labels (category, dept, state, store) as static features | Structural context no univariate model can see |

**5.5A** is the Code With Me baseline. **5.5B** and **5.5C** are Watch Only reference configurations you run in 5.9A and 5.9B for comparison.

Adding lags in a notebook is a one-liner. Maintaining a governed feature pipeline that recomputes these lags daily across 100,000 SKUs — with late data, backfills, and a product hierarchy lookup — is a data engineering project that takes months.

---
## 5.4 — Load LightGBM Parameters
**[Watch Only]**

In [ ]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

lgb_params["random_state"] = RANDOM_SEED

print(f"Loading params from: {params_file}")
for k, v in lgb_params.items():
    print(f"  {k:<22}: {v}")

---
## 5.5A — Configure MLForecast (Base)
**[Code With Me — 3 lines]**

Fill in `lags`, `lag_transforms`, and `date_features`. Use weekly, biweekly, and monthly lags. Rolling mean on lag 7 with a 28-day window. Day-of-week and month as date features.

Sections 5.5B and 5.5C define richer configs — you will compare all three in 5.9A and 5.9B.

In [ ]:
mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=__FILL_IN__,                    # [7, 14, 28]
    lag_transforms={
        7: __FILL_IN__,                  # [RollingMean(window_size=28)]
    },
    date_features=__FILL_IN__,           # ["dayofweek", "month"]
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {next(iter(mlf.models.values())).__class__.__name__}")

---
### 5.5B — Feature Engineering Reference: Enhanced Numeric and Date Features
**[Watch Only]**

Walk through each feature family:
- **More lags**: catch demand echoes at 1, 2, 3, 4 weeks + monthly
- **Rolling statistics**: mean captures level, std captures volatility, min/max capture range, quantile captures distributional shape
- **Fourier features**: encode seasonality as smooth sin/cos waves rather than integer labels
- **Seasonal rolling mean**: average of the same weekday over the past N weeks

**Why Fourier over integer day-of-week?**
Integer labels (`dayofweek=3`) are ordinal — the model can learn a non-linear mapping, but only if it sees enough data.
Fourier features encode the circular structure of the week *in the input*, so the model needs fewer samples to learn the pattern.
On sparse series or short histories, Fourier features often outperform integer encoding.

The base config in 5.5A is deliberately minimal. This block shows the full numeric and date toolkit. 5.5C adds categorical static features on top.

In [ ]:
# 5.5b — Feature Engineering Reference: Richer MLForecast Feature Sets
# This cell is a REFERENCE — swap the mlf definition in 5.5 to experiment.
# Each feature family is shown independently, then combined at the bottom.

import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import (
    RollingMean,
    RollingStd,
    RollingMin,
    RollingMax,
    RollingQuantile,
    SeasonalRollingMean,
)

# ── 1. Lag list ───────────────────────────────────────────────────────────────
# Explicit lags: 1 week, 2 weeks, 3 weeks, 4 weeks, ~1 month
LAGS = [7, 14, 21, 28]

# ── 2. Rolling statistics on lag 7 ───────────────────────────────────────────
# window_size=4 weeks → statistics over a full month of same-weekday observations
# RollingQuantile(p=0.5) is the rolling median — robust to outlier demand spikes
ROLLING_28 = [
    RollingMean(window_size=28),
    RollingStd(window_size=28),
    RollingMin(window_size=28),
    RollingMax(window_size=28),
    RollingQuantile(p=0.25, window_size=28),  # lower quartile
    RollingQuantile(p=0.75, window_size=28),  # upper quartile
]

# ── 3. Seasonal rolling mean ──────────────────────────────────────────────────
# SeasonalRollingMean(season_length=7, window_size=4):
#   average of the same weekday over the past 4 weeks.
#   Directly encodes the weekly demand profile without the model needing to
#   discover it from raw lags.
SEASONAL_ROLLING = [
    SeasonalRollingMean(season_length=7, window_size=4),   # 4-week same-weekday mean
    SeasonalRollingMean(season_length=7, window_size=8),   # 8-week same-weekday mean
]

# ── 4. Fourier date features (via MLForecast date_features callables) ─────────
# MLForecast's date_features accepts any named callable that takes a
# pandas DatetimeIndex and returns a numeric array.
# Two Fourier terms per season (k=1) encode the weekly cycle as a smooth wave.
# k=2 adds a second harmonic — captures asymmetric within-week patterns.
#
# Why this instead of dayofweek integer?
# Integer encoding: model must learn non-linearity from data.
# Fourier encoding: circular structure is baked into the feature — fewer samples needed.

def fourier_sin1_weekly(dates):
    """First Fourier sine term for weekly seasonality (period=7)."""
    return np.sin(2 * np.pi * 1 * dates.dayofweek / 7)

def fourier_cos1_weekly(dates):
    """First Fourier cosine term for weekly seasonality (period=7)."""
    return np.cos(2 * np.pi * 1 * dates.dayofweek / 7)

def fourier_sin2_weekly(dates):
    """Second Fourier sine term — captures asymmetric weekly patterns."""
    return np.sin(2 * np.pi * 2 * dates.dayofweek / 7)

def fourier_cos2_weekly(dates):
    """Second Fourier cosine term — captures asymmetric weekly patterns."""
    return np.cos(2 * np.pi * 2 * dates.dayofweek / 7)

def fourier_sin1_annual(dates):
    """First Fourier sine term for annual seasonality (period=365.25)."""
    return np.sin(2 * np.pi * 1 * dates.dayofyear / 365.25)

def fourier_cos1_annual(dates):
    """First Fourier cosine term for annual seasonality (period=365.25)."""
    return np.cos(2 * np.pi * 1 * dates.dayofyear / 365.25)

DATE_FEATURES = [
    "month",
    fourier_sin1_weekly,
    fourier_cos1_weekly,
    fourier_sin2_weekly,
    fourier_cos2_weekly,
    fourier_sin1_annual,
    fourier_cos1_annual,
]

# ── 5. Full config ────────────────────────────────────────────────────────────
mlf_rich = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

# Inspect the feature set without fitting
feature_names = mlf_rich.ts.transforms
print("Rich MLForecast feature set:")
print(f"  Lags                 : {LAGS}")
print(f"  Lag-7 transforms     : {len(ROLLING_28 + SEASONAL_ROLLING)} features")
print(f"    Rolling stats      : RollingMean, RollingStd, RollingMin, RollingMax, Q25, Q75")
print(f"    Seasonal rolling   : SeasonalRollingMean x2 (4-week, 8-week same-weekday)")
print(f"  Lag-14 transforms    : RollingMean(28)")
print(f"  Date features        : {len(DATE_FEATURES)} features")
print(f"    Calendar           : month")
print(f"    Fourier weekly k=2 : sin1, cos1, sin2, cos2")
print(f"    Fourier annual k=1 : sin1, cos1")
print()
print("To use this config for CV, replace `mlf` with `mlf_rich` in cell 5.6.")
print("Expected: +1-3 wMAPE improvement on datasets with strong weekly + annual seasonality.")

---
### 5.5C — Static Categorical Features: M5 Product Hierarchy
**[Watch Only]**

Every model so far — Naive, SeasonalNaive, AutoETS, Chronos — processes each series in isolation. LightGBM is the first model in our stack that can use structural information that *spans* series.

The M5 unique_id encodes a full product-store hierarchy. We extract it and pass it as **static features** — the same value for every row of a given series. MLForecast passes them to LightGBM as constant columns at every training row.

Key points:
- `static_features` is passed to `cross_validation()` as a keyword argument — not to the constructor
- LightGBM requires numeric inputs — we integer-encode each label with pandas `.cat.codes`
- In production, static features come from a product master table, not from parsing series IDs

In [ ]:
# 5.5C — Static Categorical Features: extracted from M5 unique_id hierarchy
# ─────────────────────────────────────────────────────────────────────────────
# M5 unique_id format: {CATEGORY}_{DEPT_NUM}_{ITEM_NUM}_{STATE_ID}_{STORE_NUM}
# Example: FOODS_2_360_WI_2
#   → category = FOODS   → dept = FOODS_2   → state = WI   → store = WI_2

def add_m5_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract and integer-encode M5 hierarchy labels from unique_id.
    LightGBM requires numeric inputs — pandas Categorical codes provide stable integer encoding.
    """
    parts = df["unique_id"].str.split("_")
    df = df.copy()
    df["cat_id"]   = parts.str[0]                    # FOODS, HOBBIES, HOUSEHOLD
    df["dept_id"]  = parts.str[:2].str.join("_")     # FOODS_1, FOODS_2, HOBBIES_1 ...
    df["state_id"] = parts.str[3]                    # CA, TX, WI
    df["store_id"] = parts.str[3:5].str.join("_")   # CA_1, CA_2, TX_1 ...
    for col in ["cat_id", "dept_id", "state_id", "store_id"]:
        df[col] = df[col].astype("category").cat.codes
    return df

STATIC_FEATURES = ["cat_id", "dept_id", "state_id", "store_id"]

micro_cat = add_m5_categoricals(micro)

print("Categorical feature extraction:")
print(f"  Panel rows preserved : {len(micro_cat):,}")
for col in STATIC_FEATURES:
    vals = sorted(micro_cat[col].unique().tolist())
    print(f"  {col:<12} : {vals}  ({len(vals)} distinct encoded values)")
print()

# mlf_cat: same rich feature set as 5.5B + static categoricals at CV time
mlf_cat = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

print("MLForecast-Cat configured.")
print(f"  Numeric/date features : same as 5.5B")
print(f"  Static features       : {STATIC_FEATURES}")
print(f"  Usage: mlf_cat.cross_validation(df=micro_cat, ..., static_features=STATIC_FEATURES)")

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

In [ ]:
%%time
# Cross-validation on micro subset
# Target runtime: < 30 seconds on Colab CPU
# PredictionIntervals requires refit=True — controlled via USE_INTERVALS in config.py

cv_ml_micro = mlf_rich.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)

print(f"ML CV complete. Shape: {cv_ml_micro.shape}")
print(f"Columns: {list(cv_ml_micro.columns)}")
print(cv_ml_micro.head(3).to_string())

**Expected output:**
```
ML CV complete. Shape: (4200, 7)
Columns: ['unique_id', 'ds', 'cutoff', 'y', 'LGBMRegressor', 'LGBMRegressor-lo-80', 'LGBMRegressor-hi-80']
```

MLForecast returns interval columns directly when `prediction_intervals` is set. The reshape in 5.7 renames them to the schema format.

---
## 5.7 — Reshape MLForecast CV Output
**[Code With Me — 2 lines]**

MLForecast returns wide format with one column per model, plus interval columns named `{model}-lo-80` and `{model}-hi-80` when `prediction_intervals` is set. These come from Nixtla's `PredictionIntervals` — calibrated using held-out CV windows, not in-sample residuals.

Fill in the two rename keys for `lo_80` and `hi_80`.

In [ ]:
def reshape_mlforecast_cv(
    cv_df: pd.DataFrame,
    model_col: str,
    stage: str,
    model_name: str = "LightGBM",
) -> pd.DataFrame:
    """
    Reshape MLForecast wide CV output to forecast schema.
    Interval columns come from PredictionIntervals — Nixtla-native conformal calibration.

    Parameters
    ----------
    cv_df      : Wide CV output from MLForecast.cross_validation()
    model_col  : Name of the point forecast column (e.g. "LGBMRegressor")
    stage      : Stage label ("ml")
    model_name : Display name in the model column
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={
        model_col:              "y_hat",
        f"{model_col}-lo-80":  __FILL_IN__,   # "lo_80" — rename native interval column
        f"{model_col}-hi-80":  __FILL_IN__,   # "hi_80" — rename native interval column
    })
    df["model"] = model_name
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)
    df["lo_80"] = df["lo_80"].clip(lower=0)
    df["hi_80"] = df["hi_80"].clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


# cv_ml_micro came from mlf_rich.cross_validation() in 5.6 → model_name="LightGBM-Rich"
ml_micro = reshape_mlforecast_cv(cv_ml_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Rich")

print(f"Reshaped ML forecasts: {ml_micro.shape}")
print(f"Columns: {list(ml_micro.columns)}")
print(f"Interval sample (lo_80, y_hat, hi_80):")
print(ml_micro[["lo_80", "y_hat", "hi_80"]].describe().round(2).to_string())

**Expected output:**
```
Reshaped ML forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 5.8 — Plot: LightGBM vs Best Baseline
**[Watch Only]**

In [ ]:
# Compare LightGBM vs SeasonalNaive on a single series
sample_uid = top_series[0]
sample_cut = ml_micro["cutoff"].unique()[-1]

actuals   = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
plot_start = sample_cut - pd.Timedelta(days=42)

lgbm_fcast = ml_micro[
    (ml_micro["unique_id"] == sample_uid) &
    (ml_micro["cutoff"]    == sample_cut)
].set_index("ds")

try:
    baseline_micro_path = ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet"
    baseline_micro = pd.read_parquet(baseline_micro_path)
    snaive_fcast = baseline_micro[
        (baseline_micro["unique_id"] == sample_uid) &
        (baseline_micro["cutoff"]    == sample_cut) &
        (baseline_micro["model"]     == "SeasonalNaive")
    ].set_index("ds")
    has_baseline = True
except Exception:
    has_baseline = False

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals[actuals.index >= plot_start].index,
        actuals[actuals.index >= plot_start].values,
        color="#333", linewidth=1.0, label="Actual", zorder=3)
ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
        color="#7B1FA2", linewidth=1.5, linestyle="--", label="LightGBM", zorder=4)
ax.fill_between(lgbm_fcast.index, lgbm_fcast["lo_80"], lgbm_fcast["hi_80"],
                alpha=0.15, color="#7B1FA2", label="LightGBM 80% interval")
if has_baseline and len(snaive_fcast) > 0:
    ax.plot(snaive_fcast.index, snaive_fcast["y_hat"],
            color="#1E88E5", linewidth=1.2, linestyle=":", label="SeasonalNaive", zorder=3)
ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
ax.set_title(f"LightGBM vs SeasonalNaive — {sample_uid} (Window 3)", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
**[Watch Only]**

This cell runs all three configs on the 50-series micro subset. The comparison table shows that 50 series is not enough to reliably distinguish three feature sets — the ordering here is likely noise. Section 5.9B shows the full-panel result, which is the one to use for decisions.

In [ ]:
%%time
# 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
# ─────────────────────────────────────────────────────────────
# mlf_rich already ran in 5.6 → reuse cv_ml_micro (reshaped in 5.7 → ml_micro).
# This cell runs mlf (base) and mlf_cat on micro and compares all three.

def _get_score(scores_df, metric):
    row = scores_df[scores_df["metric"] == metric]
    return row.iloc[0]["score"] if len(row) > 0 else float("nan")

# ── mlf_rich result from 5.6/5.7 — validate and score ────────────────────────
ml_rich_micro_val = validate_forecast(ml_micro, artifact_name="05_ml_rich_micro")
scores_rich = score_forecasts(ml_rich_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Base config (mlf) on micro ────────────────────────────────────────────────
cv_base_micro = mlf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)
ml_base_micro = reshape_mlforecast_cv(cv_base_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM")
ml_base_micro_val = validate_forecast(ml_base_micro, artifact_name="05_ml_base_micro")
scores_base = score_forecasts(ml_base_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Categorical config (mlf_cat) on micro ─────────────────────────────────────
cv_cat_micro = mlf_cat.cross_validation(
    df=micro_cat,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
    static_features=STATIC_FEATURES,
)
ml_cat_micro = reshape_mlforecast_cv(cv_cat_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Cat")
ml_cat_micro_val = validate_forecast(ml_cat_micro, artifact_name="05_ml_cat_micro")
scores_cat = score_forecasts(ml_cat_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Comparison table ───────────────────────────────────────────────────────────
W = 24
print(f"Micro Subset Comparison ({MICRO_SUBSET_N} series, {N_WINDOWS} windows):")
print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
print(f"  {'-'*66}")
for label, scores in [
    ("Base — 5.5A",        scores_base),
    ("Rich — 5.5B",        scores_rich),
    ("Categorical — 5.5C", scores_cat),
]:
    wmape  = _get_score(scores, "wMAPE")
    bias   = _get_score(scores, "Bias")
    iscore = _get_score(scores, "IntervalScore_80")
    print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")
print()
print(f"Key point: {MICRO_SUBSET_N} series is not enough to reliably distinguish these configs.")
print("Any ordering here is likely noise. See 5.9B for the full-panel view.")

---
## 5.9B — Full Panel Comparison: Base vs Rich vs Categorical
**[Watch Only]**

Load all three precomputed full-panel artifacts (1,000 series). Compare the ordering here with 5.9A — the full-panel result is the reliable one. Categorical features add structural context that no univariate model can access.

In [ ]:
# 5.9B — Full Panel Comparison: Base vs Rich vs Categorical (Precomputed)
# ─────────────────────────────────────────────────────────────────────────

try:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")

    subset_label     = f"workshop_{WORKSHOP_SUBSET_N}"
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

    W = 24
    print(f"Full Panel Comparison ({WORKSHOP_SUBSET_N:,} series, {N_WINDOWS} windows — RELIABLE):")
    print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
    print(f"  {'-'*66}")
    for label, scores in [
        ("Base — 5.5A",        scores_full_base),
        ("Rich — 5.5B",        scores_full_rich),
        ("Categorical — 5.5C", scores_full_cat),
    ]:
        wmape  = _get_score(scores, "wMAPE")
        bias   = _get_score(scores, "Bias")
        iscore = _get_score(scores, "IntervalScore_80")
        print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")

    print()
    print("Key lesson: micro result is noise. Full panel is the signal.")
    print("Categorical features add structural context no univariate model can access.")

except FileNotFoundError as e:
    print(f"Artifact not found: {e}")
    print("Run: python src/build_offline_artifacts.py --stages ml ml_rich ml_cat")

**Expected output:**
```
Full Panel Comparison (1,000 series, 3 windows — RELIABLE):
  Config                  wMAPE      Bias  IntervalScore_80
  ────────────────────────────────────────────────────────
  Base — 5.5A            0.XXXX   +0.XXXX           XX.XXXX
  Rich — 5.5B            0.XXXX   +0.XXXX           XX.XXXX
  Categorical — 5.5C     0.XXXX   +0.XXXX           XX.XXXX
```

---
## 5.10 — Full Leaderboard: All ML Configs vs Baselines
**[Watch Only]**

Build the leaderboard with all three LightGBM configs alongside the statistical baselines. This shows the full progression: baseline → numeric features → categorical features.

In [ ]:
# 5.10 — Build the full leaderboard: all three ML configs + statistical baselines
# ─────────────────────────────────────────────────────────────────────────────
# Reuse artifacts already loaded in 5.9B. Load baselines fresh.

subset_label = f"workshop_{WORKSHOP_SUBSET_N}"

# Load baselines
baseline_full   = load_checkpoint("04_baseline_forecasts")
baseline_scores = score_forecasts(baseline_full, subset_name=subset_label)

# ML scores — reuse from 5.9B if in scope, otherwise reload
try:
    _base_ok = "scores_full_base" in dir() and scores_full_base is not None
except NameError:
    _base_ok = False

if not _base_ok:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([baseline_scores, scores_full_base, scores_full_rich, scores_full_cat])

print("Updated leaderboard — all models (sorted by wMAPE):")
display_cols = ["model", "wMAPE", "Bias"]
wmape_lb = (
    leaderboard_5[display_cols]
    .dropna(subset=["wMAPE"])
    .sort_values("wMAPE")
    if "wMAPE" in leaderboard_5.columns
    else leaderboard_5
)
print(wmape_lb.to_string(index=False))

**Expected output:**
```
Updated leaderboard — all models (sorted by wMAPE):
          model    wMAPE      Bias
   LightGBM-Cat   0.XXXX   +0.XXXX
  LightGBM-Rich   0.XXXX   +0.XXXX
       LightGBM   0.XXXX   +0.XXXX
        AutoETS   0.XXXX   +0.XXXX
Chronos-t5-mini   0.XXXX   -0.XXXX
  SeasonalNaive   0.XXXX   -0.XXXX
          Naive   0.XXXX   +0.XXXX
```

---
## 5.11 — Save Micro ML Artifacts
**[Watch Only]**

Save all three micro artifacts for optional take-home analysis. The full-subset artifacts were already loaded from precomputed checkpoints.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

micro_artifacts = {
    "05_ml_base_micro_forecasts.parquet": ml_base_micro_val,
    "05_ml_rich_micro_forecasts.parquet": ml_rich_micro_val,
    "05_ml_cat_micro_forecasts.parquet":  ml_cat_micro_val,
}
for fname, df_val in micro_artifacts.items():
    path = ARTIFACT_DIR / fname
    df_val.to_parquet(path, index=False)
    print(f"  ✓ Micro artifact saved : {fname} ({len(df_val):,} rows)")

print()
print("Full-panel artifacts (precomputed checkpoints):")
print(f"  ✓ 05_ml_forecasts.parquet          ({len(ml_full_base):,} rows) — LightGBM base")
print(f"  ✓ 05_ml_rich_forecasts.parquet      ({len(ml_full_rich):,} rows) — LightGBM-Rich")
print(f"  ✓ 05_ml_cat_forecasts.parquet       ({len(ml_full_cat):,} rows)  — LightGBM-Cat")

---
## 5.12 — The Enterprise Cliff
**[Watch Only]**

Adding lags in a notebook is a one-liner. **Orchestrating a scalable feature pipeline across 100,000 SKUs — with categorical lookups, backfills, and audit trails — is where enterprise forecasting systems diverge from standalone scripts.**

What we simplified:

**Lag computation at scale:**  
MLForecast computes lags on the fly during CV. In production, lags are computed once, stored in a versioned feature store, and served to the model at inference time. Late-arriving data invalidates precomputed lags — your pipeline needs a backfill strategy.

**Categorical features in production:**  
We extracted categoricals from the series ID. In production, static features come from a product master table or store hierarchy in your ERP. The feature pipeline owns the join — and must handle new products, discontinued SKUs, and store openings that weren't in the training data. A model that has never seen a new store's state encoding will return a silent garbage prediction.

**Conformal intervals at scale:**  
`PredictionIntervals` uses held-out cross-validation windows to calibrate the conformal bounds — calibration data is separate from training data. That said, the calibration set is still drawn from the same historical distribution. On series with sudden trend shifts or promotional events, the historical error distribution will not match future errors, and the intervals will undercover.

**The ROI question:**  
LightGBM-Cat beat AutoETS by some margin on wMAPE. The question your CFO asks is: how many basis points of inventory reduction does that margin translate to? If the answer is less than the annual engineering cost of maintaining a feature pipeline with a product hierarchy lookup, AutoETS is the right production choice.